# Homework 4: Optional Final Project (A+ Grade Bump)

## Parliamentary Attention and Stance toward Skilled Immigration in Japan, 2012–2024

- **Student Name:** [replace with official name before submission]
- **Policy relevance:** Japan increasingly relies on foreign workers while immigration policy remains politically contested. Measuring how often legislators discuss immigration and how those speeches are framed helps identify when policy attention rises and whether opposition is broad or concentrated.
- **Central hypothesis:** Parliamentary attention to immigration-related issues increased over the study period, but annual attention is event-driven rather than a smooth linear trend.
- **Scope:** Japanese National Diet speeches from 2012–2024, with stance and speaker-concentration summaries from the completed Skilled-Immigration project.
- **Original sources:** National Diet Library Diet Minutes API, speaker metadata, and the project's completed LLM stance-classification pipeline.


## Reproducibility note

The original pipeline contains a 351 MB text corpus and a multi-hour LLM classification stage requiring API credentials. To keep this homework runnable and below the course repository's file-size limit, this submission uses committed aggregate snapshots produced by the completed pipeline. The modular scripts below reproduce the cleaning, figures, descriptive results, and regression from those snapshots.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the course repository.')


PROJECT_ROOT = find_project_root()
SCRIPT_DIR = PROJECT_ROOT / 'src' / 'final_project' / 'skilled_immigration'
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

print(f'Project root: {PROJECT_ROOT}')


## 1. Load Data

`data.py` loads four committed aggregate snapshots: annual speech counts, overall stance counts, speaker-concentration statistics, and the prefecture-level correlation result. These are compact outputs of the original data-acquisition and LLM pipeline.


In [ ]:
import data

raw = data.run(PROJECT_ROOT)
raw['annual'].head()


### Data acquisition details

- **Primary source:** National Diet Library Diet Minutes API
- **Period:** January 2012–December 2024
- **Immigration-related keywords:** 技能実習, 外国人労働者, 出入国管理, 特定技能, 外国人材, 不法滞在, 多文化共生, 移民政策, 外国人犯罪
- **Raw pipeline scale:** 10,961 speeches and approximately 20,741 extracted text windows
- **Homework snapshots:** Four CSV files committed under `data/final_project/skilled_immigration/raw/`


## 2. Manipulate Data

`manipulate.py` validates schemas and types, removes duplicate years, orders stance categories, calculates stance percentages, creates regression variables, and writes clean outputs to the processed-data directory.


In [ ]:
import manipulate

clean = manipulate.run(PROJECT_ROOT)
clean['annual']


In [ ]:
clean['stance']


### Preprocessing summary

- Annual counts are sorted and checked for duplicate years.
- `year_centered` equals 0 in 2012 and is used in the trend regression.
- Stance counts are converted to shares of all classified speeches.
- The source project's large raw corpus is intentionally excluded because it exceeds the course repository size limit.


## 3. Visualize Data

`graph.py` generates three publication-ready figures and saves them in `notebooks/hw/final_project/figures/`.


In [ ]:
import graph
from IPython.display import Image, display

figure_paths = graph.run(PROJECT_ROOT)
for figure_path in figure_paths:
    display(Image(filename=str(figure_path)))


### Visualizations and observations

- **Annual speech volume:** Attention is highly uneven. The largest spikes occur in 2018 and 2019, while 2024 also shows high activity. This pattern suggests policy events matter more than a smooth year-by-year increase.
- **Stance distribution:** Neutral speeches form the majority (about 65%), followed by anti (about 21%) and pro (about 14%).
- **Speaker concentration:** Anti-immigration speech is not highly concentrated nationally, but within a typical prefecture the top one to three legislators account for a very large share. Local discourse may therefore reflect a small number of especially active politicians.


## 4. Model Data

The model estimates a simple OLS regression:

`annual speeches_t = beta_0 + beta_1 × (year_t - 2012) + error_t`

The dependent variable is annual immigration-related speech volume. The independent variable is the centered year. Conventional OLS standard errors are reported for this small exploratory annual series.


In [ ]:
import model

result = model.run(PROJECT_ROOT)
print(result.summary())


In [ ]:
import pandas as pd

coefficient_table = pd.DataFrame({
    'coefficient': result.params,
    'std_error': result.bse,
    'p_value': result.pvalues,
})
coefficient_table


### Model interpretation

The estimated time trend is approximately **+60 speeches per year**, but it is not statistically significant at conventional levels (**p ≈ 0.30; R² ≈ 0.10; n = 13**). The evidence therefore does not support a smooth linear increase. Instead, the descriptive figures show sharp event-driven peaks, especially around the 2018 immigration-policy reforms.

A supplementary prefecture-level analysis from the completed project found almost no association between foreign-resident share and anti-immigration speech share (**Pearson r = −0.050, p = 0.7489, n = 46 prefectures**). This indicates that exposure to foreign residents alone does not explain geographic differences in parliamentary rhetoric.


## Conclusion

Japanese parliamentary attention to immigration is episodic rather than steadily increasing. Most classified speeches are neutral, and the prefecture-level correlation with foreign-resident share is essentially zero. The clearest structural finding is concentration: national opposition is spread across many legislators, while individual prefectures are often dominated by only one or two anti-immigration speakers. For policymakers and researchers, this means local rhetoric should not automatically be interpreted as broad constituent opinion.


## Run the complete pipeline

From the repository root:

```bash
uv run python src/final_project/skilled_immigration/pipeline.py
```
